<a href="https://colab.research.google.com/github/BlazeBot8/3d-cube-grid-subdivision/blob/main/3D_subdivision_algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import plotly.graph_objects as go
import numpy as np
import itertools

# --- Grid dimensions ---
M, N, P = 2, 2, 2
STEPS = 3   # how many bisection steps to run (3 = back to cubes at half size)

def pt(p):
    # round so the same point always gives the same tuple
    return tuple(round(float(c), 9) for c in p)

def dist(a, b):
    return np.linalg.norm(np.subtract(a, b))

# --- 1. Q0: split every cube into 6 tetrahedra around its long diagonal ---
tets = []
for x, y, z in itertools.product(range(M), range(N), range(P)):
    for order in itertools.permutations(range(3)):
        corner = [x, y, z]
        path = [pt(corner)]
        for axis in order:
            corner[axis] += 1
            path.append(pt(corner))
        tets.append(tuple(path))

# --- 2. One step: cut every tet at the midpoint of its longest edge ---
def longest_edge(t):
    return max(itertools.combinations(t, 2), key=lambda e: dist(*e))

def bisect(tets):
    new_tets = []
    for t in tets:
        a, b = longest_edge(t)
        m = pt((np.array(a) + np.array(b)) / 2)
        rest = [p for p in t if p not in (a, b)]
        new_tets.append(tuple(rest + [a, m]))
        new_tets.append(tuple(rest + [b, m]))
    return new_tets

# --- 3. Cells: group tets that share a longest edge (the spine), get outline edges ---
def plane_of(tri):
    a, b, c = (np.array(p) for p in tri)
    n = np.cross(b - a, c - a)
    n = n / np.linalg.norm(n)
    if n[np.argmax(np.abs(n))] < 0:
        n = -n
    return tuple(np.round(np.append(n, n @ a), 6))

def cell_edges(tets):
    cells = {}
    for t in tets:
        cells.setdefault(tuple(sorted(longest_edge(t))), []).append(t)

    edges = set()
    for cell in cells.values():
        # boundary triangles of the cell = tet faces used only once
        count = {}
        for t in cell:
            for f in itertools.combinations(sorted(t), 3):
                count[f] = count.get(f, 0) + 1
        boundary = [f for f, c in count.items() if c == 1]

        # keep an edge only where two different face planes meet
        planes = {}
        for f in boundary:
            for e in itertools.combinations(f, 2):
                planes.setdefault(e, set()).add(plane_of(f))
        edges |= {e for e, pl in planes.items() if len(pl) > 1}
    return edges, len(cells)

# --- 4. Run all steps ---
levels = [tets]
for _ in range(STEPS):
    levels.append(bisect(levels[-1]))

# --- 5. Plot each step: previous level in gray, current level highlighted ---
def lines(edges):
    xs, ys, zs = [], [], []
    for p1, p2 in edges:
        xs.extend([p1[0], p2[0], None])
        ys.extend([p1[1], p2[1], None])
        zs.extend([p1[2], p2[2], None])
    return xs, ys, zs

for k in range(1, STEPS + 1):
    old_edges, _ = cell_edges(levels[k - 1])
    new_edges, n_cells = cell_edges(levels[k])
    old_nodes = {p for t in levels[k - 1] for p in t}
    new_nodes = {p for e in new_edges for p in e}
    added = new_nodes - old_nodes   # points created by this step's cuts

    fig = go.Figure()

    # Q(k-1) edges as background
    x, y, z = lines(old_edges)
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines',
                               line=dict(color='gray', width=1),
                               name=f'Q{k-1} Cells'))

    # Q(k) edges
    x, y, z = lines(new_edges)
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines',
                               line=dict(color='orange', width=2),
                               name=f'Q{k} Cells'))

    # Q(k) nodes that already existed
    kept = new_nodes - added
    fig.add_trace(go.Scatter3d(x=[p[0] for p in kept], y=[p[1] for p in kept], z=[p[2] for p in kept],
                               mode='markers', marker=dict(size=2, color='lightgray'),
                               name='Existing Nodes'))

    # Q(k) nodes added by this step (midpoints of longest edges)
    fig.add_trace(go.Scatter3d(x=[p[0] for p in added], y=[p[1] for p in added], z=[p[2] for p in added],
                               mode='markers', marker=dict(size=3, color='green'),
                               name=f'Q{k} New Nodes (Longest-Edge Midpoints)'))

    fig.update_layout(
        title=f'Q_{k} Partition for a {M}x{N}x{P} Cube Grid (Q{k-1} in background)',
        scene=dict(
            xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
            aspectmode='cube',
            xaxis=dict(nticks=M+2, range=[-0.5, M+0.5]),
            yaxis=dict(nticks=N+2, range=[-0.5, N+0.5]),
            zaxis=dict(nticks=P+2, range=[-0.5, P+0.5])
        ),
        height=800,
        showlegend=True
    )
    fig.show()

    print(f"Step {k}: {len(levels[k])} tets ({len(levels[k]) // (M*N*P)} per cube), "
          f"{n_cells} cells, {len(added)} new nodes, {len(new_edges)} cell edges.")

Step 1: 96 tets (12 per cube), 36 cells, 8 new nodes, 118 cell edges.


Step 2: 192 tets (24 per cube), 54 cells, 36 new nodes, 232 cell edges.


Step 3: 384 tets (48 per cube), 64 cells, 54 new nodes, 300 cell edges.
